# Phase 3 — Perplexity Baselines and Quality Degradation Alerts Evaluation

This runbook/notebook verifies the integration of perplexity scoring, dynamic 4-weight composite score renormalization, YAML-based hot-reloading baseline verification, and early-warning quality degradation alerts suppression gates.

In [ ]:
# 1. Import composite scoring and baseline verification rules
from handlers.span_quality.composite_scorer import compute_composite
from handlers.span_quality.types import ScoreMap
from handlers.span_quality.baseline_logic import should_alert_degradation, update_baseline_ewma

print("Scoring modules successfully imported.")

### 2. Verify Composite Renormalization (4-Weight System with Perplexity)
We evaluate how the composite score adapts when perplexity is present (weight = 0.10) vs when it is absent/null (renormalizing across the other 3 weights).

In [ ]:
# Baseline weights: Coherence (0.25), Faithfulness (0.35), Toxicity (0.15), Perplexity (0.10)
scores_with_perplexity = ScoreMap(coherence=0.8, toxicity=0.1, faithfulness=0.9, perplexity=12.0)
comp_val_4w, weights_4w = compute_composite(scores_with_perplexity)

print(f"4-Weight Composite Quality Score: {comp_val_4w:.4f}")
print("Normalized weights used:", {k: round(v, 4) for k, v in weights_4w.items()})

# Fallback weights (Perplexity is None)
scores_no_perplexity = ScoreMap(coherence=0.8, toxicity=0.1, faithfulness=0.9)
comp_val_3w, weights_3w = compute_composite(scores_no_perplexity)

print(f"\nFallback 3-Weight Composite Quality Score: {comp_val_3w:.4f}")
print("Normalized weights used:", {k: round(v, 4) for k, v in weights_3w.items()})

### 3. Verify Degradation Alerting Logic & Threshold Gates
Checks if `should_alert_degradation` fires when the current 1-hour window average drops below 90% of the baseline average.

In [ ]:
baseline_avg = 0.85
under_threshold_avg = 0.75  # < 90% of 0.85 (0.765)
healthy_avg = 0.82          # > 90% of 0.85

alert_under = should_alert_degradation(under_threshold_avg, baseline_avg, sample_count=25)
alert_healthy = should_alert_degradation(healthy_avg, baseline_avg, sample_count=25)
alert_low_samples = should_alert_degradation(under_threshold_avg, baseline_avg, sample_count=10)

print(f"Alert triggered for average={under_threshold_avg} (samples=25): {alert_under}")
print(f"Alert triggered for average={healthy_avg} (samples=25): {alert_healthy}")
print(f"Alert triggered for average={under_threshold_avg} (samples=10): {alert_low_samples}")